In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/2026-dm-lab-2-private-competition/sampleSubmission.csv
/kaggle/input/competitions/2026-dm-lab-2-private-competition/data_identification.csv
/kaggle/input/competitions/2026-dm-lab-2-private-competition/final_posts.json
/kaggle/input/competitions/2026-dm-lab-2-private-competition/emotion.csv


In [2]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("/kaggle/input/competitions/2026-dm-lab-2-private-competition")

# 1. Load tabular metadata & labels
sample_sub = pd.read_csv(DATA_DIR / "sampleSubmission.csv")
data_id = pd.read_csv(DATA_DIR / "data_identification.csv")
emotions = pd.read_csv(DATA_DIR / "emotion.csv")

print("sampleSubmission columns:", sample_sub.columns.tolist(), "shape:", sample_sub.shape)
print("data_identification columns:", data_id.columns.tolist(), "shape:", data_id.shape)
print("emotion columns:", emotions.columns.tolist(), "shape:", emotions.shape)

# 2. Parse final_posts.json (handles list of dicts, nested {'root': {'_source': {'post': ...}}}, or {id: text} dict)
with open(DATA_DIR / "final_posts.json", "r", encoding="utf-8") as f:
    raw_json = json.load(f)

records = []
if isinstance(raw_json, dict):
    for pid, val in raw_json.items():
        if isinstance(val, str):
            records.append({"post_id": str(pid), "raw_text": val})
        elif isinstance(val, dict):
            text = val.get("text") or val.get("post_text") or val.get("content") or ""
            records.append({"post_id": str(val.get("post_id", val.get("id", pid))), "raw_text": text})
elif isinstance(raw_json, list):
    for item in raw_json:
        if isinstance(item, dict):
            # Unwrap nested Elasticsearch/Twitter-style wrapper if present
            node = item.get("root", item)
            node = node.get("_source", node) if isinstance(node, dict) else item
            node = node.get("post", node) if isinstance(node, dict) else item
            pid = node.get("post_id") or node.get("id") or item.get("post_id") or item.get("id")
            text = node.get("text") or node.get("post_text") or node.get("content") or ""
            records.append({"post_id": str(pid), "raw_text": str(text)})

posts_df = pd.DataFrame(records).drop_duplicates(subset=["post_id"])
print("Parsed posts_df shape:", posts_df.shape)

# 3. Standardize ID column names & merge train / test splits
id_col_data = data_id.columns[0]
split_col = data_id.columns[1]
data_id[id_col_data] = data_id[id_col_data].astype(str)

id_col_emo = emotions.columns[0]
label_col = emotions.columns[1]
emotions[id_col_emo] = emotions[id_col_emo].astype(str)

merged = data_id.merge(posts_df, left_on=id_col_data, right_on="post_id", how="left")
merged["raw_text"] = merged["raw_text"].fillna("")

train_df = (
    merged[merged[split_col].str.lower() == "train"]
    .merge(emotions, left_on=id_col_data, right_on=id_col_emo, how="inner")
    .reset_index(drop=True)
)

# Preserve exact sampleSubmission row order for test_df
sub_id_col, sub_target_col = sample_sub.columns[0], sample_sub.columns[1]
sample_sub[sub_id_col] = sample_sub[sub_id_col].astype(str)
test_df = sample_sub[[sub_id_col]].merge(
    merged, left_on=sub_id_col, right_on=id_col_data, how="left"
).reset_index(drop=True)
test_df["raw_text"] = test_df["raw_text"].fillna("")

# 4. Social media text cleaning (preserves emotional cues, splits CamelCase hashtags, separates punctuation)
def clean_social_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    # Split hashtags like #SoGross -> So Gross
    text = re.sub(r"#([A-Za-z0-9_]+)", lambda m: " " + re.sub(r"([a-z])([A-Z])", r"\1 \2", m.group(1)) + " ", text)
    # Normalize repeated characters beyond 2 (e.g. "sooooo" -> "soo")
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    # Separate exclamation/question marks so they become standalone tokens
    text = re.sub(r"([!?]+)", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df["raw_text"].apply(clean_social_text)
test_df["clean_text"] = test_df["raw_text"].apply(clean_social_text)

print(f"\nTrain shape: {train_df.shape} | Test shape: {test_df.shape}")
print("Class distribution:\n", train_df[label_col].value_counts())

sampleSubmission columns: ['post_id', 'label'] shape: (1440, 2)
data_identification columns: ['post_id', 'set'] shape: (4800, 2)
emotion columns: ['post_id', 'label'] shape: (3360, 2)
Parsed posts_df shape: (4800, 2)

Train shape: (3360, 5) | Test shape: (1440, 4)
Class distribution:
 label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64


In [3]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score

CLASSES = sorted(train_df[label_col].unique())
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for i, c in enumerate(CLASSES)}
y_train = train_df[label_col].map(label2id).values

# 1. Handcrafted Stylistic & Punctuation Features (9 features)
def extract_stylistic_features(df: pd.DataFrame) -> pd.DataFrame:
    raw = df["raw_text"].fillna("")
    clean = df["clean_text"].fillna("")
    feats = pd.DataFrame(index=df.index)
    feats["char_len"] = raw.str.len()
    feats["word_count"] = clean.str.split().str.len()
    feats["avg_word_len"] = feats["char_len"] / (feats["word_count"] + 1)
    feats["excl_count"] = raw.str.count(r"!")
    feats["quest_count"] = raw.str.count(r"\?")
    feats["upper_ratio"] = raw.apply(lambda s: sum(1 for c in s if c.isupper()) / (len(s) + 1))
    feats["all_caps_count"] = raw.str.count(r"\b[A-Z]{2,}\b")
    feats["negation_count"] = clean.str.lower().str.count(
        r"\b(not|no|never|nothing|nowhere|neither|nobody|none|n't)\b"
    )
    feats["first_person_count"] = clean.str.lower().str.count(
        r"\b(i|me|my|mine|myself|we|us|our)\b"
    )
    return feats

X_style_train_raw = extract_stylistic_features(train_df)
X_style_test_raw = extract_stylistic_features(test_df)

# 2. 5-Fold Stratified CV for Branch 1 (TF-IDF Word 1-3 + Char 3-5 + Stylistic)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lr = np.zeros((len(train_df), len(CLASSES)))
test_lr = np.zeros((len(test_df), len(CLASSES)))
oof_svc = np.zeros((len(train_df), len(CLASSES)))
test_svc = np.zeros((len(test_df), len(CLASSES)))
oof_nb = np.zeros((len(train_df), len(CLASSES)))
test_nb = np.zeros((len(test_df), len(CLASSES)))

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_train), 1):
    scaler = StandardScaler()
    st_tr = csr_matrix(scaler.fit_transform(X_style_train_raw.iloc[tr_idx]))
    st_va = csr_matrix(scaler.transform(X_style_train_raw.iloc[va_idx]))
    st_te = csr_matrix(scaler.transform(X_style_test_raw))

    word_vec = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=25000, sublinear_tf=True, strip_accents="unicode")
    w_tr = word_vec.fit_transform(train_df["clean_text"].iloc[tr_idx])
    w_va = word_vec.transform(train_df["clean_text"].iloc[va_idx])
    w_te = word_vec.transform(test_df["clean_text"])

    char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=35000, sublinear_tf=True)
    c_tr = char_vec.fit_transform(train_df["clean_text"].iloc[tr_idx])
    c_va = char_vec.transform(train_df["clean_text"].iloc[va_idx])
    c_te = char_vec.transform(test_df["clean_text"])

    X_tr_nb, X_va_nb, X_te_nb = hstack([w_tr, c_tr]).tocsr(), hstack([w_va, c_va]).tocsr(), hstack([w_te, c_te]).tocsr()
    X_tr_feat = hstack([w_tr, c_tr, st_tr]).tocsr()
    X_va_feat = hstack([w_va, c_va, st_va]).tocsr()
    X_te_feat = hstack([w_te, c_te, st_te]).tocsr()

    clf_lr = LogisticRegression(C=1.5, class_weight="balanced", max_iter=1000, random_state=42)
    clf_lr.fit(X_tr_feat, y_train[tr_idx])
    oof_lr[va_idx] = clf_lr.predict_proba(X_va_feat)
    test_lr += clf_lr.predict_proba(X_te_feat) / skf.n_splits

    clf_svc = CalibratedClassifierCV(LinearSVC(C=0.35, class_weight="balanced", random_state=42), cv=3)
    clf_svc.fit(X_tr_feat, y_train[tr_idx])
    oof_svc[va_idx] = clf_svc.predict_proba(X_va_feat)
    test_svc += clf_svc.predict_proba(X_te_feat) / skf.n_splits

    clf_nb = ComplementNB(alpha=0.5)
    clf_nb.fit(X_tr_nb, y_train[tr_idx])
    oof_nb[va_idx] = clf_nb.predict_proba(X_va_nb)
    test_nb += clf_nb.predict_proba(X_te_nb) / skf.n_splits

oof_ens = 0.45 * oof_lr + 0.40 * oof_svc + 0.15 * oof_nb
test_ens = 0.45 * test_lr + 0.40 * test_svc + 0.15 * test_nb
print(f"Branch 1 (TF-IDF + Stylistic) 5-Fold OOF Macro-F1: {f1_score(y_train, oof_ens.argmax(axis=1), average='macro'):.4f}")

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

Branch 1 (TF-IDF + Stylistic) 5-Fold OOF Macro-F1: 0.5136


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [4]:
import gc
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running Transformer fine-tuning on:", device)
assert device.type == "cuda", "Turn on GPU T4/P100 in Kaggle Session Options -> Accelerator first!"

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True, max_length=max_len, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

test_ds = EmotionDataset(test_df["raw_text"].fillna(""))
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

oof_trf = np.zeros((len(train_df), len(CLASSES)))
test_trf = np.zeros((len(test_df), len(CLASSES)))

EPOCHS = 4
BATCH_SIZE = 32
LR = 2e-5

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_train), 1):
    torch.manual_seed(42 + fold)
    np.random.seed(42 + fold)

    train_ds = EmotionDataset(train_df["raw_text"].iloc[tr_idx].fillna(""), y_train[tr_idx])
    val_ds = EmotionDataset(train_df["raw_text"].iloc[va_idx].fillna(""), y_train[va_idx])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    # Smooth inverse-sqrt class weights to lift minority classes (disgust, fear)
    raw_cw = compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=y_train[tr_idx])
    smooth_cw = torch.tensor(np.sqrt(raw_cw), dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=smooth_cw, label_smoothing=0.05)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(CLASSES), ignore_mismatched_sizes=True
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )

    best_val_f1 = -1.0
    best_val_probs = None
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            outputs = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            loss = criterion(outputs.logits, batch["labels"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        model.eval()
        val_probs = []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                ).logits
                val_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())

        val_probs = np.vstack(val_probs)
        val_f1 = f1_score(y_train[va_idx], val_probs.argmax(axis=1), average="macro")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_probs = val_probs
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    oof_trf[va_idx] = best_val_probs
    print(f"Fold {fold} Best Twitter-RoBERTa Macro-F1: {best_val_f1:.4f}")

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    fold_test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            ).logits
            fold_test_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())

    test_trf += np.vstack(fold_test_probs) / skf.n_splits

    del model, optimizer, scheduler, best_state
    torch.cuda.empty_cache()
    gc.collect()

# 4. Blend Branch 2 (Twitter-RoBERTa) + Branch 1 (TF-IDF/MiniLM Hybrid)
trf_oof_f1 = f1_score(y_train, oof_trf.argmax(axis=1), average="macro")
print(f"\nBranch 2 (Twitter-RoBERTa alone) 5-Fold OOF Macro-F1: {trf_oof_f1:.4f}")

best_alpha, best_blend_f1 = 0.80, -1.0
for alpha in [0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    blend_probs = alpha * oof_trf + (1 - alpha) * oof_ens
    score = f1_score(y_train, blend_probs.argmax(axis=1), average="macro")
    if score > best_blend_f1:
        best_blend_f1 = score
        best_alpha = alpha

oof_final = best_alpha * oof_trf + (1 - best_alpha) * oof_ens
test_final = best_alpha * test_trf + (1 - best_alpha) * test_ens
print(f"Blended Ensemble ({best_alpha:.2f} RoBERTa + {1-best_alpha:.2f} Hybrid) OOF Macro-F1: {best_blend_f1:.4f}")

# 5. Macro-F1 class threshold optimization
res = minimize(
    lambda w: -f1_score(y_train, (oof_final * w).argmax(axis=1), average="macro"),
    np.ones(len(CLASSES)),
    method="Nelder-Mead",
    options={"maxiter": 600},
)
best_weights = res.x if -res.fun > best_blend_f1 else np.ones(len(CLASSES))

opt_oof_preds = (oof_final * best_weights).argmax(axis=1)
final_oof_f1 = f1_score(y_train, opt_oof_preds, average="macro")
print(f"Final F1-Tuned Ensemble OOF Macro-F1: {final_oof_f1:.4f}\n")
print(classification_report(y_train, opt_oof_preds, target_names=CLASSES, digits=4))

# 6. Build submission matching sampleSubmission.csv exact column names & ID order
final_test_preds = [id2label[i] for i in (test_final * best_weights).argmax(axis=1)]
submission = pd.DataFrame({
    sub_id_col: sample_sub[sub_id_col].values,
    sub_target_col: final_test_preds,
})
assert len(submission) == len(sample_sub)
assert submission[sub_target_col].isna().sum() == 0

submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv! First 5 rows:")
print(submission.head())
print("\nPredicted test label counts:")
print(submission[sub_target_col].value_counts())

Running Transformer fine-tuning on: cuda


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `3`.


... (19 outputs left out)

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([6, 768])
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([6])          

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect 

Fold 5 Best Twitter-RoBERTa Macro-F1: 0.5776

Branch 2 (Twitter-RoBERTa alone) 5-Fold OOF Macro-F1: 0.5940
Blended Ensemble (0.65 RoBERTa + 0.35 Hybrid) OOF Macro-F1: 0.6010
Final F1-Tuned Ensemble OOF Macro-F1: 0.6040

              precision    recall  f1-score   support

       anger     0.5738    0.6113    0.5920       674
     disgust     0.4745    0.3899    0.4281       454
        fear     0.6489    0.5903    0.6182       288
         joy     0.7842    0.7600    0.7719       875
     sadness     0.6445    0.6497    0.6471       628
    surprise     0.5209    0.6213    0.5667       441

    accuracy                         0.6268      3360
   macro avg     0.6078    0.6037    0.6040      3360
weighted avg     0.6279    0.6268    0.6259      3360


Saved submission.csv! First 5 rows:
   post_id     label
0  xyzx9jp  surprise
1  l3vzzxt     anger
2  g680iug   sadness
3  m4t8bp6       joy
4  ai3al6e  surprise

Predicted test label counts:
label
joy         381
anger       315
sadnes

In [5]:
import pandas as pd

sub = pd.read_csv("/kaggle/working/submission.csv")
sub.columns = ["id", "label"]
sub.to_csv("/kaggle/working/submission.csv", index=False)

!head -n 5 /kaggle/working/submission.csv
print("\nShape:", sub.shape, "| Nulls:", sub.isna().sum().to_dict())

id,label
xyzx9jp,surprise
l3vzzxt,anger
g680iug,sadness
m4t8bp6,joy

Shape: (1440, 2) | Nulls: {'id': 0, 'label': 0}
